# 방법 C 보충 실험 — ERR_FACT_QTY 기반 pseudo-label 후보 검증

**목적:** 박상은(방법 B)의 Phase 1 EDA에서 발견한 `ERR_FACT_QTY` 컬럼이 방법 C의 pseudo-defect 선별에 도움이 되는지 검증한다.

**배경:** `ERR_FACT_QTY`는 unlabeled에만 있는 컬럼으로, 값이 0보다 큰 행이 전체의 76.7%다. 처음에는 이것이 불량 발생 기록일 가능성이 있어 pseudo-defect 후보로 활용할 수 있을지 실험했다.

**결론 미리 보기:** ERR_FACT_QTY는 샷별 불량 여부가 아니라 배치 단위 수량이다. 이 필터를 추가해도 방법 C 전략 D의 성능이 개선되지 않는다.

---
| 항목 | 값 |
|---|---|
| 비교 대상 | 기존 방법 C 전략 D (CN7/RG3+생산상태 필터, 39,870행) |
| 신규 시도 | 동일 필터 + ERR_FACT_QTY > 0 (33,639행) |
| 평가 방법 | align_score + 5-fold CV ROC/PR (Wilcoxon 검정) |

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import roc_auc_score, average_precision_score
from imblearn.over_sampling import SMOTE
from scipy.stats import wilcoxon

DATA_DIR = Path('../data/raw')
SPLIT_DIR = Path('../data/splits')

lab = pd.read_csv(DATA_DIR / 'labeled_data.csv')
unl = pd.read_csv(DATA_DIR / 'unlabeled_data.csv')
lab['PassOrFail'] = lab['PassOrFail'].map({'Y': 0, 'N': 1})

DROP = [
    'Mold_Temperature_1','Mold_Temperature_2','Mold_Temperature_5',
    'Mold_Temperature_6','Mold_Temperature_7','Mold_Temperature_8',
    'Mold_Temperature_9','Mold_Temperature_10','Mold_Temperature_11',
    'Mold_Temperature_12','Barrel_Temperature_7','PART_FACT_SERIAL'
]
FEAT = [c for c in lab.select_dtypes('number').columns
        if c not in DROP and c != 'PassOrFail']

X_lab = lab[FEAT].fillna(0).values
y_lab = lab['PassOrFail'].values
mask_d = y_lab == 1
mask_n = y_lab == 0

folds = [np.load(SPLIT_DIR / f'fold_{i}.npy', allow_pickle=True).item() for i in range(5)]

print(f'labeled: {len(lab):,}행  (불량 {mask_d.sum()}, 양품 {mask_n.sum()})')
print(f'unlabeled: {len(unl):,}행')
print(f'피처: {len(FEAT)}개')

## 1. ERR_FACT_QTY 분포 확인

이 컬럼이 샷(shot)별 불량 여부인지, 아니면 다른 의미인지 분포로 먼저 확인한다.

labeled_data의 불량률은 0.89%다. 만약 ERR_FACT_QTY > 0이 샷별 불량을 의미한다면
unlabeled에서도 비슷한 비율이어야 한다.

In [ ]:
eq = pd.to_numeric(unl['ERR_FACT_QTY'], errors='coerce').fillna(0)

print('=== ERR_FACT_QTY 분포 (unlabeled 전체) ===')
print(f'ERR_FACT_QTY = 0  : {(eq==0).sum():>9,}행  ({(eq==0).mean()*100:.1f}%)')
print(f'ERR_FACT_QTY > 0  : {(eq>0).sum():>9,}행  ({(eq>0).mean()*100:.1f}%)')
print()
print('값 분포 상위 15개:')
print(eq.value_counts().head(15).to_string())
print()
print('--- 해석 ---')
print(f'labeled 불량률: 0.89%')
print(f'ERR>0 비율:     {(eq>0).mean()*100:.1f}%')
print()
print('ERR>0 비율이 76.7%인데 실제 불량률은 0.89%다.')
print('값도 3, 2, 5, 12, 130 같은 정수 — 배치/시프트 단위 수량으로 보인다.')
print('개별 샷의 불량 여부가 아니다.')

## 2. 필터 조합별 align_score 비교

**align_score**: 선별한 pseudo-defect 집합이 labeled 불량과 얼마나 같은 방향으로 이탈하는지 측정한다.
- `dir_match`: 25개 피처 중 불량과 같은 방향으로 이탈한 피처 비율 (50% = 무작위 수준)
- `cosine`: 이탈 방향의 코사인 유사도 (1 = 완전 일치, 0 = 무관, -1 = 반대)

기존 방법 C 전략 D는 kNN으로 top-50을 선택해서 dir_match=100%, cosine=0.879였다.

In [ ]:
sc_sel = StandardScaler().fit(X_lab)
Xl_sc = sc_sel.transform(X_lab)
defect_dir_sc = Xl_sc[mask_d].mean(0) - Xl_sc[mask_n].mean(0)
normal_mean_sc = Xl_sc[mask_n].mean(0)

unl_feat = unl[[c for c in FEAT if c in unl.columns]].reindex(columns=FEAT, fill_value=0)
mask_cn7rg3 = unl['PART_NAME'].str.contains('CN7|RG3', na=False)
mask_equip  = unl['EQUIP_CD'].isin(['S01','S12','S14'])
mask_prod   = (
    (pd.to_numeric(unl['Barrel_Temperature_1'], errors='coerce') >= 200) &
    (pd.to_numeric(unl['Average_Screw_RPM'],    errors='coerce') > 0)
)
mask_err = eq > 0

def pool_align(X_raw, label):
    Xp_sc = sc_sel.transform(X_raw.fillna(0).values if hasattr(X_raw, 'fillna') else X_raw)
    pv  = Xp_sc.mean(0) - normal_mean_sc
    dv  = defect_dir_sc
    dm  = float(np.mean((pv * dv) > 0))
    cos = float(np.dot(dv, pv) / (np.linalg.norm(dv) * np.linalg.norm(pv) + 1e-9))
    print(f'  {label:<50}: dir={dm:.0%}  cos={cos:+.3f}  n={len(X_raw):,}')
    return dm, cos

print('=== 필터 조합별 전체 풀 align_score ===')
print('  (풀 전체 평균 기준 — 선택 전 품질 가늠용)\n')
pool_align(lab[FEAT][mask_d],                                             '[기준] labeled 불량 71개')
pool_align(unl_feat[mask_cn7rg3 & mask_equip & mask_prod],                '[기존 방법C] CN7/RG3+설비+생산상태')
pool_align(unl_feat[mask_cn7rg3 & mask_equip & mask_prod & mask_err],     '[신규] 기존 필터 + ERR>0')
pool_align(unl_feat[mask_equip & mask_prod & (unl['EQUIP_CD']=='S14')],   '[박상은 S14] S14+생산상태')
print()
print('기존 방법C 필터 풀보다 ERR>0 추가 풀의 cosine이 0.454 → 0.593으로 높아진다.')
print('그러나 풀 전체 평균이지, kNN이 실제로 선택하는 top-50의 품질은 다를 수 있다.')

## 3. Strategy D kNN 선택 후 align_score 비교

풀 전체 평균이 아니라, 실제로 kNN이 labeled 불량과 가장 가까운 top-50을 뽑았을 때의 품질을 확인한다.

In [ ]:
X_unl_A = unl_feat[mask_cn7rg3 & mask_equip & mask_prod].values
X_unl_B = unl_feat[mask_cn7rg3 & mask_equip & mask_prod & mask_err].values

print(f'풀 A (기존 방법C 필터)       : {len(X_unl_A):,}행')
print(f'풀 B (기존 필터 + ERR>0)     : {len(X_unl_B):,}행')
print()

def top_n_knn(X_unl, n=50):
    X_unl_sc = sc_sel.transform(X_unl)
    knn = NearestNeighbors(n_neighbors=min(n, len(X_unl_sc)), metric='euclidean')
    knn.fit(Xl_sc[mask_d])
    dists, _ = knn.kneighbors(X_unl_sc)
    return X_unl[np.argsort(dists.min(axis=1))[:n]]

top50_A = top_n_knn(X_unl_A, 50)
top50_B = top_n_knn(X_unl_B, 50)

print('=== kNN top-50 선택 후 align_score ===')
pool_align(lab[FEAT][mask_d],  '[기준] labeled 불량 71개')
pool_align(top50_A,            '[풀A] 기존 방법C top-50')
pool_align(top50_B,            '[풀B] ERR>0 추가 top-50')
print()
print('풀 전체 평균은 B가 높았지만,')
print('ERR>0 조건이 labeled 불량과 가장 가까운 샘플들을 걸러내 버린다.')
print('결과적으로 top-50의 cosine이 0.899 → 0.534로 떨어진다.')

## 4. 5-fold CV 성능 비교

실제로 분류기(RF) 학습에 각 풀의 top-50을 투입했을 때 ROC-AUC와 PR-AUC가 어떻게 달라지는지 측정한다.

In [ ]:
def run_cv(X_pseudo, seed=42):
    roc_list, pr_list = [], []
    for fold in folds:
        tr_idx, val_idx = fold['train'], fold['val']
        X_tr_l, y_tr_l = X_lab[tr_idx], y_lab[tr_idx]
        X_val, y_val   = X_lab[val_idx], y_lab[val_idx]
        X_tr_aug = np.vstack([X_tr_l, X_pseudo])
        y_tr_aug = np.concatenate([y_tr_l, np.ones(len(X_pseudo), dtype=int)])
        sc_f = StandardScaler().fit(X_tr_l)
        X_tr_sc  = sc_f.transform(X_tr_aug)
        X_val_sc = sc_f.transform(X_val)
        k_sm = max(1, min(5, int(y_tr_aug.sum()) - 1))
        try:
            X_res, y_res = SMOTE(k_neighbors=k_sm, random_state=seed).fit_resample(X_tr_sc, y_tr_aug)
        except Exception:
            X_res, y_res = X_tr_sc, y_tr_aug
        clf = RandomForestClassifier(n_estimators=200, random_state=seed, n_jobs=-1)
        clf.fit(X_res, y_res)
        prob = clf.predict_proba(X_val_sc)[:, 1]
        roc_list.append(roc_auc_score(y_val, prob))
        pr_list.append(average_precision_score(y_val, prob))
    return np.array(roc_list), np.array(pr_list)

def run_baseline(seed=42):
    roc_list, pr_list = [], []
    for fold in folds:
        tr_idx, val_idx = fold['train'], fold['val']
        X_tr, y_tr   = X_lab[tr_idx], y_lab[tr_idx]
        X_val, y_val = X_lab[val_idx], y_lab[val_idx]
        sc_f = StandardScaler().fit(X_tr)
        X_tr_sc, X_val_sc = sc_f.transform(X_tr), sc_f.transform(X_val)
        k_sm = max(1, min(5, int(y_tr.sum()) - 1))
        X_res, y_res = SMOTE(k_neighbors=k_sm, random_state=seed).fit_resample(X_tr_sc, y_tr)
        clf = RandomForestClassifier(n_estimators=200, random_state=seed, n_jobs=-1)
        clf.fit(X_res, y_res)
        prob = clf.predict_proba(X_val_sc)[:, 1]
        roc_list.append(roc_auc_score(y_val, prob))
        pr_list.append(average_precision_score(y_val, prob))
    return np.array(roc_list), np.array(pr_list)

print('실행 중... (약 1~2분)')
roc_base, pr_base = run_baseline()
roc_A,    pr_A    = run_cv(top50_A)
roc_B,    pr_B    = run_cv(top50_B)

_, p_A = wilcoxon(roc_A - roc_base, alternative='greater') if any(roc_A != roc_base) else (None, 1.0)
_, p_B = wilcoxon(roc_B - roc_base, alternative='greater') if any(roc_B != roc_base) else (None, 1.0)

print()
print('=== 5-fold CV 결과 (RF, n_pseudo=50) ===')
print(f'Baseline (labeled only)      : ROC={roc_base.mean():.4f}  PR={pr_base.mean():.4f}')
print(f'Strategy D 풀A (기존)        : ROC={roc_A.mean():.4f} ({roc_A.mean()-roc_base.mean():+.4f})  '
      f'PR={pr_A.mean():.4f} ({pr_A.mean()-pr_base.mean():+.4f})  p={p_A:.4f}')
print(f'Strategy D 풀B (ERR>0 추가)  : ROC={roc_B.mean():.4f} ({roc_B.mean()-roc_base.mean():+.4f})  '
      f'PR={pr_B.mean():.4f} ({pr_B.mean()-pr_base.mean():+.4f})  p={p_B:.4f}')
print()
print(f'풀B - 풀A ROC 차이 (fold별): {np.round(roc_B - roc_A, 4).tolist()}')
print(f'평균 ROC 차이: {(roc_B-roc_A).mean():+.4f}  (5fold 중 {(roc_B > roc_A).sum()}개만 B가 우세)')

## 5. 결론

**ERR_FACT_QTY 필터는 방법 C에 추가할 의미가 없다.**

이유 두 가지:

**1. ERR_FACT_QTY는 샷별 불량이 아니다.**  
unlabeled에서 76.7%가 ERR_FACT_QTY > 0이다. labeled의 불량률이 0.89%인 동일 공정에서 76.7%가 불량일 수는 없다. 값도 3, 5, 12, 130처럼 크다 — 배치 또는 시프트 단위로 집계된 수량이다.

**2. kNN top-50 품질이 오히려 떨어진다.**  
전체 풀 평균 cosine은 0.454 → 0.593으로 올라가지만, ERR>0 조건이 labeled 불량과 가장 가까운 샘플들을 제거한다. kNN이 실제로 선택하는 top-50의 cosine이 0.899 → 0.534로 내려간다. 5-fold CV ROC도 5개 fold 중 4개에서 하락한다. Wilcoxon p=0.50 — 개선 근거 없음.

**방법 B(K-Means)에 대한 함의:**  
ERR_FACT_QTY는 방법 C pseudo-labeling에는 도움이 안 되지만, 방법 B에서는 다른 방식으로 활용 가능하다. ERR_FACT_QTY=0인 행(생산 이상 없는 기간)을 정상 클러스터 검증 기준으로 쓰거나, ERR_FACT_QTY 분포를 이상 점수의 외부 참조값으로 활용하는 것은 별도로 검토할 수 있다.

---
수치 요약 (results/tables/method_c_errqty_experiment.csv 참조):

| 조건 | pool n | ROC | ΔROC | Wilcoxon p | top-50 cosine |
|---|---|---|---|---|---|
| Baseline | — | 0.9271 | — | — | — |
| Strategy D 풀A (기존) | 39,870 | 0.9493 | +0.022 | 0.156 | 0.899 |
| Strategy D 풀B (ERR>0) | 33,639 | 0.9399 | +0.013 | 0.500 | 0.534 |